![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/books_sales.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql
select * from current_books

In [0]:
from pyspark.sql import functions as F

def process_books_sales():
    orders_df = (spark.readStream.table("orders_silver").withColumn("book", F.explode("books")))

    books_df = (spark.read.table("current_books"))

    query = (

        orders_df
        .join(books_df, books_df.book_id == orders_df.book.book_id,"inner")
        .writeStream
            .outputMode("append")
            .option("checkpointLocation", f"{bookstore.checkpoint_path}/books_sales")
            .trigger(availableNow=True)
            .table("books_sales")
    )
    query.awaitTermination()


In [0]:
process_books_sales()

In [0]:
%sql
select * from books_sales

In [0]:
#bookstore.load_new_data()
bookstore.process_bronze()
bookstore.process_books_silver()
bookstore.process_current_books()

In [0]:
process_books_sales()

In [0]:
%sql
select * from books_sales

In [0]:
bookstore.process_orders_silver() 
process_books_sales()

In [0]:
%sql
select *
from books_sales